# Дообучение PP-OCRv5 rec (eslav) под русские чеки

> **Только Google Colab (GPU T4).** Не запускайте ячейки локально в Cursor/VS Code/Jupyter на Mac — не будет `/content`, `nvidia-smi` и `paddlepaddle-gpu`.
>
> Открыть: [colab.research.google.com](https://colab.research.google.com) → File → Upload notebook → этот файл → Runtime → Change runtime type → **T4 GPU**.

Дообучает `eslav_PP-OCRv5_mobile_rec` на **смеси** синтетических строк и кропов из датасета [cdek-ocr/receipt-ocr-ru](https://huggingface.co/datasets/cdek-ocr/receipt-ocr-ru), затем экспортирует ONNX для `Skinemsya_ml`.

**Локально перед Colab (уже сделано, если есть `data/cdek_synth.zip`):**

```bash
python -m training.synthgen --count 2000 --out data/synth
python -m training.cdek.export_rec_lines --synth data/synth --out data/rec_mix
cd data/rec_mix && zip -r ../cdek_synth.zip lines rec_gt.txt && cd ../..
```

Подробности: `docs/training/cdek-receipt-ocr-ru.md`.

Ожидаемое время в Colab: установка ~5 мин, обучение 30–90 мин на T4.


In [ ]:
# Guard: этот ноутбук только для Google Colab.
import os, sys
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or os.path.isdir("/content")
if not IN_COLAB:
    raise SystemExit(
        "❌ Это не Google Colab.\n"
        "Откройте ноутбук на https://colab.research.google.com\n"
        "(File → Upload notebook) и выберите Runtime → T4 GPU.\n"
        "Локально на Mac обучение не поддерживается (нет CUDA / paddlepaddle-gpu)."
    )
print("OK: Google Colab")
!nvidia-smi


In [ ]:
# Шаг 2. Устанавливаем PaddlePaddle (GPU) и клонируем PaddleOCR (~5 минут).
# Если pip не находит paddlepaddle-gpu под текущую CUDA, посмотрите актуальную
# команду на https://www.paddlepaddle.org.cn/en/install/quick
%cd /content
!pip install -q "paddlepaddle-gpu" -i https://www.paddlepaddle.org.cn/packages/stable/cu126/ || pip install -q paddlepaddle-gpu
!git clone --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git
%cd /content/PaddleOCR
!pip install -q -r requirements.txt
!pip install -q paddle2onnx

import paddle
print("Paddle:", paddle.__version__, "| CUDA доступна:", paddle.device.is_compiled_with_cuda())

In [ ]:
# Шаг 3. Загружаем датасет строк (cdek_synth.zip или synth.zip).
# Кнопка «Выбрать файлы» — выберите архив с lines/ + rec_gt.txt.
from google.colab import files

uploaded = files.upload()  # cdek_synth.zip предпочтительно
zip_name = next(iter(uploaded))

# --- Вариант с Google Drive ---
# from google.colab import drive
# drive.mount('/content/drive')
# zip_name = '/content/drive/MyDrive/cdek_synth.zip'

!mkdir -p /content/dataset
!unzip -qo "{zip_name}" -d /content/dataset
# zip мог положить files в корень или в подпапку
import os, pathlib, shutil
root = pathlib.Path('/content/dataset')
if not (root / 'rec_gt.txt').exists():
    for candidate in root.rglob('rec_gt.txt'):
        parent = candidate.parent
        for item in parent.iterdir():
            dest = root / item.name
            if dest.exists():
                continue
            shutil.move(str(item), str(dest))
        break
!ls /content/dataset && wc -l /content/dataset/rec_gt.txt


In [ ]:
# Шаг 4. Делим датасет на train/val (95/5) и находим словарь символов eslav.
import glob
import pathlib
import random

lines = pathlib.Path("/content/dataset/rec_gt.txt").read_text(encoding="utf-8").splitlines()
random.Random(42).shuffle(lines)
split = int(len(lines) * 0.95)
pathlib.Path("/content/dataset/rec_gt_train.txt").write_text("\n".join(lines[:split]) + "\n", encoding="utf-8")
pathlib.Path("/content/dataset/rec_gt_val.txt").write_text("\n".join(lines[split:]) + "\n", encoding="utf-8")
print(f"train: {split}, val: {len(lines) - split}")

# Словарь символов модели eslav (восточнославянские языки).
dict_candidates = glob.glob("/content/PaddleOCR/ppocr/utils/dict/**/*eslav*", recursive=True)
assert dict_candidates, "Словарь eslav не найден — проверьте версию PaddleOCR"
DICT_PATH = dict_candidates[0]
print("Словарь:", DICT_PATH)

# Проверяем, что все символы датасета есть в словаре (отсутствующие модель не выучит).
dict_chars = set(pathlib.Path(DICT_PATH).read_text(encoding="utf-8").split("\n"))
dataset_chars = set()
for line in lines:
    if "\t" in line:
        dataset_chars.update(line.split("\t", 1)[1])
missing = dataset_chars - dict_chars - {" "}
print("Символы вне словаря:", missing if missing else "нет — всё покрыто")

# Конфиг обучения eslav-модели (если его нет в репо — берём базовый PP-OCRv5 mobile).
cfg_candidates = glob.glob("/content/PaddleOCR/configs/rec/**/*eslav*", recursive=True)
CONFIG_PATH = cfg_candidates[0] if cfg_candidates else "configs/rec/PP-OCRv5/PP-OCRv5_mobile_rec.yml"
print("Конфиг:", CONFIG_PATH)

PRETRAINED_URL = "https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/eslav_PP-OCRv5_mobile_rec_pretrained.pdparams"
!wget -q {PRETRAINED_URL} -O /content/eslav_pretrained.pdparams && ls -lh /content/eslav_pretrained.pdparams

In [ ]:
# Шаг 5. Обучение (~30-60 минут на T4).
# Дообучаем с претрейна: маленький learning rate, немного эпох.
# Метрика acc в логах — доля полностью верно распознанных строк на валидации.
!cd /content/PaddleOCR && python tools/train.py -c {CONFIG_PATH} \
    -o Global.pretrained_model=/content/eslav_pretrained.pdparams \
       Global.character_dict_path={DICT_PATH} \
       Global.max_text_length=40 \
       Global.epoch_num=10 \
       Global.save_model_dir=/content/output/receipt_rec \
       Global.eval_batch_step="[0, 500]" \
       Global.print_batch_step=50 \
       Optimizer.lr.learning_rate=0.0005 \
       Train.dataset.data_dir=/content/dataset \
       Train.dataset.label_file_list="['/content/dataset/rec_gt_train.txt']" \
       Train.loader.batch_size_per_card=128 \
       Eval.dataset.data_dir=/content/dataset \
       Eval.dataset.label_file_list="['/content/dataset/rec_gt_val.txt']" \
       Eval.loader.batch_size_per_card=128

In [ ]:
# Шаг 6. Экспорт лучшего чекпоинта в inference-формат и конвертация в ONNX.
!cd /content/PaddleOCR && python tools/export_model.py -c {CONFIG_PATH} \
    -o Global.pretrained_model=/content/output/receipt_rec/best_accuracy \
       Global.character_dict_path={DICT_PATH} \
       Global.max_text_length=40 \
       Global.save_inference_dir=/content/inference/receipt_rec

!ls /content/inference/receipt_rec

# Новый PIR-формат Paddle 3.x — inference.json; старый — inference.pdmodel.
import os
model_file = ("inference.json" if os.path.exists("/content/inference/receipt_rec/inference.json")
              else "inference.pdmodel")
!cd /content && paddle2onnx \
    --model_dir /content/inference/receipt_rec \
    --model_filename {model_file} \
    --params_filename inference.pdiparams \
    --save_file /content/receipt_rec.onnx \
    --opset_version 14
!ls -lh /content/receipt_rec.onnx

In [ ]:
# Шаг 7. Скачиваем результат: модель + словарь символов.
import shutil
shutil.copy(DICT_PATH, "/content/receipt_rec_keys.txt")

from google.colab import files
files.download("/content/receipt_rec.onnx")
files.download("/content/receipt_rec_keys.txt")

## Подключение модели к сервису

1. Положите файлы в `data/models/receipt_rec.onnx` и `data/models/receipt_rec_keys.txt`.
2. Бенчмарк на CDEK test (после `python -m training.cdek.download_and_prepare`):

```bash
python -m training.eval_receipts --dataset data/cdek/test
python -m training.eval_receipts --dataset data/cdek/test \
  --rec-model data/models/receipt_rec.onnx --rec-keys data/models/receipt_rec_keys.txt
```

3. Запуск с новой моделью:

```bash
ML_REC_MODEL_PATH=data/models/receipt_rec.onnx \
ML_REC_KEYS_PATH=data/models/receipt_rec_keys.txt \
uvicorn app.main:app --port 8000
```

В Docker — volume + `ML_REC_*` в `docker-compose.yml` / prod compose.
